# FSOT Biohub v50 — CPU competitive pipeline

**Kaggle is CPU-only.** Train/tune the U-Net on GPU locally; this notebook runs inference on CPU only.

Pipeline: **U-Net FT detect** → **FSOT linking** → **ML division refine (f47)** → **ILP** → **division-gap patch** → `submission.csv`

Local train-proxy (`44b6_0113de3b`): **0.979** edge score (49/0/1). Full 4-dataset run: **312,387 rows**.

**Inputs:** competition test + `cellmot-ft-detector-biohub` + `cellmot-baseline-artifacts` + `fsot-v50-competitive-bundle` + `fsot-offline-dependencies`

Lean ref: https://github.com/dappalumbo91/FSOT-2.1-Lean (commit `4bded9d`)

In [ ]:
import os
import sys
import shutil
import glob
import subprocess
import zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

# CPU-only — competition has no GPU workers
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")
os.environ.setdefault("KAGGLE_CPU_ONLY", "1")
os.environ.setdefault("CELLMOT_DEVICE", "cpu")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("TORCH_NUM_THREADS", "4")
os.environ.setdefault("CELLMOT_DET_TTA", "0")

# v50 competitive defaults (match kaggle_main_runner.py @ 4bded9d)
os.environ.setdefault("BIOHUB_ENGINE", "auto")
os.environ.setdefault("FSOT_VISION_CALIBRATE", "1")
os.environ.setdefault("FSOT_LIVING_EMERGENCE", "1")
os.environ.setdefault("FSOT_LIVING_ADAPTIVE", "1")
os.environ.setdefault("FSOT_DET_CONF_RANK", "1")
os.environ.setdefault("FSOT_LIVING_PROXY_ACCURACY", "0.90")
os.environ.setdefault("FSOT_LIVING_MIN_UNET_CONF", "0.0")
os.environ.setdefault("FSOT_LIVING_TARGET_PER_FRAME", "258")
os.environ.setdefault("FSOT_LINK_MODE", "fsot")
os.environ.setdefault("FSOT_GATE_FRAC", "0.42")
os.environ.setdefault("FSOT_GATE_ADAPTIVE", "1")
os.environ.setdefault("FSOT_GATE_RESCUE", "1")
os.environ.setdefault("CELLMOT_USE_FT", "1")
os.environ.setdefault("CELLMOT_DET_THRESHOLD", "0.48")
os.environ.setdefault("CELLMOT_EDGE_THRESHOLD", "0.25")
os.environ.setdefault("CELLMOT_NMS_UM", "6.0")
os.environ.setdefault("CELLMOT_POOL_UM", "8.0")
os.environ.setdefault("CELLMOT_USE_ILP", "1")
os.environ.setdefault("CELLMOT_ILP_MAX_EDGES", "80000")
os.environ.setdefault("CELLMOT_GRAPH_CONSISTENCY", "auto")
os.environ.setdefault("FSOT_GAP_LINK", "1")
os.environ.setdefault("FSOT_DIVISION_ML_REFINE", "1")
os.environ.setdefault("FSOT_ML_REFINE_FRAMES", "47")
os.environ.setdefault("FSOT_ML_REFINE_REPLACE_FRAMES", "47")
os.environ.setdefault("FSOT_ML_REFINE_EDGE_THRESHOLD", "0.05")
os.environ.setdefault("FSOT_ML_REFINE_MODE", "replace")
os.environ.setdefault("FSOT_ML_PRESERVE_FSOT_FRAMES", "29")
os.environ.setdefault("FSOT_ML_POST_ILP_CORRECT", "0")
os.environ.setdefault("FSOT_ML_NEAREST_DAUGHTER", "0")
os.environ.setdefault("FSOT_ML_DIVISION_GAP_PATCH", "1")
os.environ.setdefault("FSOT_MITOSIS_VELOCITY", "1")
os.environ.setdefault("KAGGLE_SUBMISSION_FAST_VALIDATE", "0")

print("Kaggle input:", os.listdir("/kaggle/input"))

_bundle_hits = glob.glob("/kaggle/input/**/kaggle_main_runner.py", recursive=True)
if _bundle_hits:
    src_dir = Path(_bundle_hits[0]).parent
    for name in [
        "kaggle_main_runner.py",
        "biohub_unet_engine.py",
        "biohub_competitive.py",
        "fsot_division_ml_refine.py",
        "fsot_cellular_bridge.py",
        "fsot_core.py",
        "fsot_living_emergence.py",
        "fsot_vision_calibrate.py",
        "fsot_original_competition.py",
        "submission_io.py",
        "validate_kaggle_submission.py",
        "csv_to_geffs.py",
        "download_ft_weights.py",
        "kaggle_wheel_bootstrap.py",
        "cellmot_code_bundle.zip",
    ]:
        p = src_dir / name
        if p.exists():
            shutil.copy2(p, WORK / name)
    print(f"Copied v50 bundle from {src_dir}")
else:
    raise FileNotFoundError(
        "fsot-v50-competitive-bundle not attached — add damianpalumbo/fsot-v50-competitive-bundle"
    )

required = [
    "biohub_unet_engine.py",
    "fsot_division_ml_refine.py",
    "kaggle_main_runner.py",
    "cellmot_code_bundle.zip",
]
missing = [n for n in required if not (WORK / n).exists()]
if missing:
    raise FileNotFoundError(f"Bundle incomplete after copy: {missing}")

_ft = glob.glob("/kaggle/input/**/cellmot-ft-detector-biohub/**/edge_predictor_best.pth", recursive=True)
_wt = _ft or glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True)
if _wt:
    os.environ["CELLMOT_UNET_WEIGHTS"] = _wt[0]
    print(f"[UNET] weights: {_wt[0]}")
else:
    raise FileNotFoundError("cellmot-ft-detector-biohub weights not found")

sys.path.insert(0, str(WORK))
from kaggle_wheel_bootstrap import extract_cellmot_bundle, install_cellmot_wheels

wheel_dir = install_cellmot_wheels()
if wheel_dir is None:
    raise RuntimeError("cellmot-baseline-artifacts wheels required")
print(f"Cellmot wheels: {wheel_dir}")
if not extract_cellmot_bundle(WORK):
    raise RuntimeError("cellmot_code_bundle.zip required")

os.environ.setdefault("BIOHUB_ENGINE", "fsot_unet")
print("[ENV] BIOHUB_ENGINE=", os.environ.get("BIOHUB_ENGINE"))
print("[ENV] FSOT_DIVISION_ML_REFINE=", os.environ.get("FSOT_DIVISION_ML_REFINE"))
print("[ENV] FSOT_ML_DIVISION_GAP_PATCH=", os.environ.get("FSOT_ML_DIVISION_GAP_PATCH"))

In [ ]:
import runpy
import sys

sys.path.insert(0, "/kaggle/working")
runpy.run_path("/kaggle/working/kaggle_main_runner.py", run_name="__main__")

import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv")
counts = sub.groupby(["dataset", "row_type"]).size()
print(counts)
print(f"submission rows: {len(sub)}")

# Local CPU parity check (2026-07-14 run)
EXPECTED_TOTAL = 312387
EXPECTED_BY_DS = {
    "44b6_0113de3b": {"node": 26099, "edge": 24549},
    "44b6_0b24845f": {"node": 55310, "edge": 48994},
    "6bba_05b6850b": {"node": 8054, "edge": 7361},
    "6bba_05db0fb1": {"node": 73871, "edge": 68149},
}
tol = 50  # allow tiny CPU/thread variance
assert abs(len(sub) - EXPECTED_TOTAL) <= tol, f"row count {len(sub)} != {EXPECTED_TOTAL}"
for ds, parts in EXPECTED_BY_DS.items():
    ds_sub = sub[sub["dataset"] == ds]
    for row_type, exp in parts.items():
        got = int((ds_sub["row_type"] == row_type).sum())
        assert abs(got - exp) <= tol, f"{ds} {row_type}: {got} != {exp}"
print("PARITY OK — matches local v50 CPU run")